In [5]:
import numpy as np
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
movies = pd.read_csv("data/tmdb_5000_movies.csv")
credits = pd.read_csv("data/tmdb_5000_credits.csv")

movies = movies.merge(credits, on='title')

In [7]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [8]:
movies.dropna(inplace=True)

In [9]:
def convert_to_list(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert_to_list)
movies['keywords'] = movies['keywords'].apply(convert_to_list)

In [10]:
def convert_cast(obj):
    L = []
    cnt = 0
    for i in ast.literal_eval(obj):
        if cnt != 3:
            L.append(i['name'])
            cnt+=1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [11]:
def get_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(get_director)

In [12]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])


In [13]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [14]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [ ]:
df = movies[['movie_id', 'title', 'tags']]
df['tags'] = df['tags'].apply(lambda x: x.lower())

In [16]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vector = cv.fit_transform(df['tags']).toarray()

In [17]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    Y = []
    for i in text.split():
        Y.append(ps.stem(i))

    return " ".join(Y)


df['tags'] = df['tags'].apply(stem)

In [19]:
similarity = cosine_similarity(vector)

In [20]:
def recommend(movie):
    idx = df[df['title'] == movie].index[0]
    dist = similarity[idx]

    movies_list = sorted(list(enumerate(dist)), reverse=True, key=lambda x: x[1])[1:6]
    
    for i in movies_list:
        print(df.iloc[i[0]].title)

In [22]:
import pickle

In [ ]:
pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [25]:
pickle.dump(df.to_dict(), open('movies_dict.pkl', 'wb'))